# 03 — Donor-marker calibration

This notebook inspects the production `controls` and `calibrate` artifacts. Calibration uses registered mutually exclusive references to estimate donor-local marker background and uncertainty. It does not fit a RESTORE/GMM maximum, normalize by positive-cell frequency, or replace the original intensity scale.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

from phenocycler.artifacts import ContractError, StageManifest, StaleArtifactError
from phenocycler.config import load_config
from phenocycler.donor_pipeline import (
    DonorStageReceipt,
    _donor_stage_config,
    donor_receipt_path,
)
from phenocycler.expression import read_single_partition
from phenocycler.marker_calibration import CalibrationConfig
from phenocycler.pipeline import STAGE_METHODS, RunContext, resolve_run_context, status

CONFIG_PATH = None
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
status_code = status(context)
print(f"status return code: {status_code}")

## Evidence semantics

Each calibrated marker keeps several deliberately distinct quantities:

- **Intensity** (`<marker>__corrected_intensity`) is the authoritative selected-expression value and retains its physical scale.
- **Log2 threshold ratio** is the signed log2 separation from the donor-marker threshold after the production `log1p` transform; zero is the threshold, positive is above it, and negative is below it.
- **Empirical tail probability** is the upper-tail null probability under clean reference controls; smaller values are stronger evidence against background.
- **Expression probability** is `1 - empirical_tail_probability`; larger values are stronger expression evidence. It is an evidence score, not a cohort frequency or an absolute cross-donor abundance.
- **State** is `positive`, `negative`, `indeterminate`, or `unavailable`. A cell is positive or negative only when its value is stable relative to the bootstrapped threshold interval and its empirical evidence agrees; otherwise uncertainty is preserved.

The model audit records control counts, contamination, threshold intervals, status, and failure reasons. Invalid donor-marker models leave measurable cells `indeterminate`; an unmeasured cell or panel-absent marker is `unavailable`. Neither becomes a guessed negative.

## Production execution

`controls` selects mutually exclusive background references from estimation-eligible cells. `calibrate` fits donor-marker models and applies them to every measurable cell while preserving analysis eligibility for typing. The pipelined production queue publishes each donor receipt atomically, so this notebook can review completed donors without competing with the active queue. Start or resume production from a shell only when another pipeline process is not already running.

In [ ]:
print("Read-only review notebook. Production command:")
print("python -m phenocycler.pipeline run --config config.ini --pipelined --through calibrate --no-export")

In [ ]:
validated_receipt_ids = set()
validated_artifacts = set()
validated_code_hashes = set()


def validate_receipt_tree(receipt):
    if receipt.content_id in validated_receipt_ids:
        return
    receipt.verify_identity()
    if receipt.method_version != STAGE_METHODS[receipt.stage]:
        raise StaleArtifactError(
            f"{receipt.stage}/{receipt.donor_id}: method version changed"
        )
    expected_root = context.output_root(receipt.stage).expanduser().resolve()
    if Path(receipt.output_root).expanduser().resolve() != expected_root:
        raise ContractError(
            f"{receipt.stage}/{receipt.donor_id}: output root differs from {expected_root}"
        )
    code_hash = receipt.code.source_sha256
    if code_hash not in validated_code_hashes:
        if receipt.code.capture_current().source_sha256 != code_hash:
            raise StaleArtifactError(
                f"{receipt.stage}/{receipt.donor_id}: producing code changed"
            )
        validated_code_hashes.add(code_hash)
    receipt.validate_current(
        config=_donor_stage_config(context, receipt.stage, receipt.donor_id),
        validate_code=False,
        validate_inputs=False,
        validation_mode="fast",
    )
    for artifact in receipt.inputs:
        validate_receipt_input(artifact)
    validated_receipt_ids.add(receipt.content_id)


def validate_receipt_input(artifact):
    cache_key = (artifact.kind, artifact.path, artifact.content_sha256)
    if cache_key in validated_artifacts:
        return
    if artifact.kind == "donor_receipt":
        upstream = DonorStageReceipt.read_json(artifact.path)
        if upstream.content_id != artifact.content_sha256:
            raise StaleArtifactError(
                f"input donor receipt changed: {artifact.name}"
            )
        validate_receipt_tree(upstream)
    else:
        artifact.validate_current(mode="fast")
    validated_artifacts.add(cache_key)


def current_stage_receipts(stage_name):
    receipts = []
    for donor in context.donors:
        path = donor_receipt_path(context, stage_name, donor)
        if not path.exists():
            continue
        receipt = DonorStageReceipt.read_json(path)
        if receipt.stage != stage_name or receipt.donor_id != str(donor):
            raise ContractError(
                f"receipt at {path} does not belong to {stage_name}/{donor}"
            )
        validate_receipt_tree(receipt)
        receipts.append(receipt)
    return tuple(receipts)


stage_manifests = {
    stage_name: (
        StageManifest.read_json(context.stage_manifest_path(stage_name))
        if context.stage_manifest_path(stage_name).exists() else None
    )
    for stage_name in ("controls", "calibrate")
}
stage_receipts = {
    stage_name: (() if stage_manifests[stage_name] is not None
                 else current_stage_receipts(stage_name))
    for stage_name in ("controls", "calibrate")
}
completed_donors = {}
progress_rows = []
for stage_name in ("controls", "calibrate"):
    path = context.stage_manifest_path(stage_name)
    manifest = stage_manifests[stage_name]
    receipts = stage_receipts[stage_name]
    receipt_donors = tuple(receipt.donor_id for receipt in receipts)
    if path.exists():
        donor_ids = tuple(str(donor) for donor in manifest.completed_donors)
        stage_status = "COMPLETE"
        method_version = manifest.method_version
        rows = manifest.output.total_rows
        content_id = manifest.content_id[:12]
    else:
        donor_ids = receipt_donors
        stage_status = "IN PROGRESS" if receipts else "NOT STARTED"
        method_version = receipts[0].method_version if receipts else None
        rows = sum(receipt.output.row_count for receipt in receipts)
        content_id = None
    completed_donors[stage_name] = donor_ids
    progress_rows.append({
        "stage": stage_name,
        "status": stage_status,
        "donors": f"{len(donor_ids)}/{len(context.donors)}",
        "rows available": rows,
        "method_version": method_version,
        "cohort content": content_id,
    })
display(pd.DataFrame(progress_rows))

control_donors = set(completed_donors["controls"])
calibrated_donors = set(completed_donors["calibrate"])
donor_progress = pd.DataFrame({
    "donor_id": context.donors,
    "calibration progress": [
        "calibrated" if donor in calibrated_donors
        else "ready for calibration" if donor in control_donors
        else "upstream processing"
        for donor in context.donors
    ],
})
display(donor_progress)

## Model audits

Inspect validity and uncertainty before interpreting per-cell evidence. The calibration audit is one row per donor-marker model; the control audit records how reference controls were selected.

In [ ]:
def read_available_audit(stage_name, aggregate_name):
    aggregate_path = context.config.audit_dir / f"{aggregate_name}.parquet"
    if stage_manifests[stage_name] is not None and aggregate_path.exists():
        return pd.read_parquet(aggregate_path), "sealed cohort audit"
    frames = []
    for receipt in stage_receipts[stage_name]:
        parquet_sidecars = [
            Path(sidecar.path)
            for sidecar in receipt.sidecars
            if Path(sidecar.path).suffix == ".parquet"
        ]
        if len(parquet_sidecars) != 1:
            raise RuntimeError(
                f"{stage_name}/{receipt.donor_id} must have one audit Parquet sidecar"
            )
        frames.append(pd.read_parquet(parquet_sidecars[0]))
    if not frames:
        return pd.DataFrame(), "not available"
    return (
        pd.concat(frames, ignore_index=True),
        f"{len(frames)} validated donor receipt(s)",
    )


control_audit, control_audit_source = read_available_audit(
    "controls", "reference_control_models"
)
calibration_audit, calibration_audit_source = read_available_audit(
    "calibrate", "marker_calibration_models"
)

if not control_audit.empty:
    print(f"control models: {len(control_audit):,} ({control_audit_source})")
    display(control_audit.head(20))
else:
    print("No completed donor control audits are available yet.")

if not calibration_audit.empty:
    print(f"calibration models: {len(calibration_audit):,} ({calibration_audit_source})")
    display(calibration_audit.groupby("status", dropna=False).size().rename("donor_markers").to_frame())
    audit_columns = [
        column for column in (
            "donor_id", "marker", "reference", "status", "status_reason",
            "marker_alpha", "n_control_candidates", "n_controls_selected",
            "n_controls_clean", "contamination_fraction",
            "threshold_raw", "threshold_ci_low_raw", "threshold_ci_high_raw"
        ) if column in calibration_audit
    ]
    display(calibration_audit.loc[:, audit_columns].head(30))
else:
    print("No completed donor calibration audits are available yet.")

## Cohort calibration diagnostics

The status matrix localizes failed donor-marker models and distinguishes them from donors that have not completed calibration. The control plot shows the two configured hard gates: enough clean controls and no more than the permitted contamination. Bootstrap threshold width is descriptive rather than a release cutoff; a wide interval identifies models whose uncertainty is more likely to leave cells indeterminate and therefore deserves review. None of these plots uses positive-cell frequency or cross-donor uniformity as a calibration target.

In [ ]:
STATUS_ORDER = (
    "valid",
    "insufficient_controls",
    "insufficient_tail_resolution",
    "excess_control_contamination",
    "invalid_background",
)
STATUS_COLORS = {
    "not_completed": "#d9d9d9",
    "valid": "#2a9d8f",
    "insufficient_controls": "#e9c46a",
    "insufficient_tail_resolution": "#f4a261",
    "excess_control_contamination": "#e76f51",
    "invalid_background": "#8d5a97",
}

if calibration_audit.empty:
    print("No completed donor calibration audits are available to plot yet.")
else:
    calibration_plot = calibration_audit.copy()
    calibration_plot["donor_id"] = calibration_plot["donor_id"].astype(str)
    donor_order = list(context.donors)
    marker_order = [marker.name for marker in context.registry.active_markers]
    unknown_statuses = sorted(
        set(calibration_plot["status"].dropna()).difference(STATUS_ORDER)
    )
    if unknown_statuses:
        raise ContractError(f"unknown calibration statuses: {unknown_statuses}")

    status_matrix = (
        calibration_plot.pivot(index="donor_id", columns="marker", values="status")
        .reindex(index=donor_order, columns=marker_order)
    )
    status_codes = np.zeros(status_matrix.shape, dtype=int)
    for code, status_name in enumerate(STATUS_ORDER, start=1):
        status_codes[status_matrix.eq(status_name).to_numpy()] = code
    status_names = ("not_completed", *STATUS_ORDER)
    status_cmap = ListedColormap([STATUS_COLORS[name] for name in status_names])
    status_norm = BoundaryNorm(
        np.arange(-0.5, len(status_names) + 0.5),
        status_cmap.N,
    )

    fig, ax = plt.subplots(
        figsize=(max(13, 0.48 * len(marker_order)), max(6, 0.32 * len(donor_order))),
        constrained_layout=True,
    )
    ax.imshow(status_codes, aspect="auto", cmap=status_cmap, norm=status_norm)
    ax.set_xticks(np.arange(len(marker_order)), labels=marker_order, rotation=60, ha="right")
    ax.set_yticks(np.arange(len(donor_order)), labels=donor_order)
    ax.set_xlabel("Marker")
    ax.set_ylabel("Donor")
    ax.set_title(f"Donor-marker calibration status ({calibration_audit_source})")
    for donor_index, marker_index in np.argwhere(status_codes > 1):
        ax.text(marker_index, donor_index, "×", ha="center", va="center", color="black", fontweight="bold")
    status_handles = [
        Patch(facecolor=STATUS_COLORS[name], label=name.replace("_", " "))
        for name in status_names
    ]
    ax.legend(handles=status_handles, title="Model status", loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.show()

    calibration_config = CalibrationConfig()
    fig, axes = plt.subplots(
        1, 2, figsize=(17, max(6, 0.32 * len(donor_order))),
        gridspec_kw={"width_ratios": (0.8, 1.35)},
        constrained_layout=True,
    )

    control_margin = calibration_plot["n_controls_clean"] - calibration_plot["min_controls"]
    contamination_percent = 100.0 * calibration_plot["contamination_fraction"]
    for status_name in STATUS_ORDER:
        selected = calibration_plot["status"].eq(status_name)
        finite = selected & control_margin.notna() & contamination_percent.notna()
        if finite.any():
            axes[0].scatter(
                control_margin.loc[finite],
                contamination_percent.loc[finite],
                s=30 if status_name == "valid" else 75,
                marker="o" if status_name == "valid" else "X",
                color=STATUS_COLORS[status_name],
                edgecolor="black" if status_name != "valid" else "none",
                alpha=0.75,
                label=status_name.replace("_", " "),
            )
    axes[0].axvline(0, color="#444444", linestyle="--", linewidth=1.2, label="minimum clean controls")
    axes[0].axhline(
        100.0 * calibration_config.max_contamination_fraction,
        color="#b22222", linestyle="--", linewidth=1.2,
        label="maximum contamination",
    )
    axes[0].set_yscale("symlog", linthresh=0.1)
    axes[0].set_ylim(bottom=0)
    axes[0].set_xlabel("Clean controls above configured minimum")
    axes[0].set_ylabel("Selected-control contamination (%)")
    axes[0].set_title("Hard control-quality gates")
    axes[0].grid(True, which="both", alpha=0.2)
    axes[0].legend(fontsize=8, loc="best")
    for row_index in calibration_plot.index[calibration_plot["status"].ne("valid")]:
        if pd.notna(control_margin.loc[row_index]) and pd.notna(contamination_percent.loc[row_index]):
            row = calibration_plot.loc[row_index]
            axes[0].annotate(
                f"{row['donor_id']} / {row['marker']}",
                (control_margin.loc[row_index], contamination_percent.loc[row_index]),
                xytext=(5, -8), textcoords="offset points", fontsize=8, va="top",
            )

    calibration_plot["threshold_ci_span_log2"] = (
        calibration_plot["threshold_ci_high_log1p"]
        - calibration_plot["threshold_ci_low_log1p"]
    ) / np.log(2.0)
    uncertainty_matrix = (
        calibration_plot.pivot(
            index="donor_id", columns="marker", values="threshold_ci_span_log2"
        ).reindex(index=donor_order, columns=marker_order)
    )
    uncertainty_values = uncertainty_matrix.to_numpy(dtype=float)
    finite_uncertainty = uncertainty_values[np.isfinite(uncertainty_values)]
    if finite_uncertainty.size:
        uncertainty_vmax = float(np.quantile(finite_uncertainty, 0.95))
        uncertainty_vmax = max(uncertainty_vmax, np.finfo(float).eps)
        uncertainty_cmap = plt.get_cmap("magma").copy()
        uncertainty_cmap.set_bad(STATUS_COLORS["not_completed"])
        image = axes[1].imshow(
            np.ma.masked_invalid(uncertainty_values),
            aspect="auto", cmap=uncertainty_cmap, vmin=0, vmax=uncertainty_vmax,
        )
        colorbar = fig.colorbar(image, ax=axes[1], shrink=0.8)
        colorbar.set_label("95% CI span on log2(1 + intensity) scale")
        for donor_index, marker_index in np.argwhere(status_codes > 1):
            axes[1].text(marker_index, donor_index, "×", ha="center", va="center", color="black", fontweight="bold")
        axes[1].legend(
            handles=[
                Patch(facecolor=STATUS_COLORS["not_completed"], label="not completed / no valid interval"),
                Line2D([0], [0], marker="x", color="black", linestyle="none", label="invalid model"),
            ],
            loc="upper left", bbox_to_anchor=(1.16, 1.0), fontsize=8,
        )
    else:
        axes[1].text(0.5, 0.5, "No valid threshold intervals yet", ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_xticks(np.arange(len(marker_order)), labels=marker_order, rotation=60, ha="right")
    axes[1].set_yticks(np.arange(len(donor_order)), labels=donor_order)
    axes[1].set_xlabel("Marker")
    axes[1].set_ylabel("Donor")
    axes[1].set_title("Bootstrap threshold uncertainty (descriptive; color saturates at cohort 95th percentile)")
    plt.show()

## Inspect one donor-marker pair

The evidence table is wide by design: every registered marker contributes its intensity, calibrated state, threshold-relative score, empirical probabilities, and repeated model provenance. Choose a donor and marker below to inspect the same quantities that downstream typing consumes.

In [ ]:
AVAILABLE_DONORS = tuple(
    donor for donor in context.donors
    if donor in calibrated_donors
)
DONOR = AVAILABLE_DONORS[0] if AVAILABLE_DONORS else None
# Replace DONOR above with another value from AVAILABLE_DONORS to inspect it.
evidence = pd.DataFrame()
calibrated_markers = []
MARKER = None

if DONOR is not None:
    DONOR = str(DONOR)
    if DONOR not in AVAILABLE_DONORS:
        raise ValueError(
            f"donor {DONOR} has no validated calibration receipt; "
            f"available donors: {AVAILABLE_DONORS}"
        )
    print(f"Inspecting donor {DONOR}; {len(AVAILABLE_DONORS)}/{len(context.donors)} donors are available.")
    evidence = read_single_partition(context.config.marker_evidence_dir, DONOR)
    calibrated_markers = [
        marker.name
        for marker in context.registry.active_markers
        if f"{marker.name}__state" in evidence
    ]
    if calibrated_markers:
        MARKER = calibrated_markers[0]
        metric_suffixes = (
            "corrected_intensity", "state", "log2_threshold_ratio",
            "empirical_tail_probability", "expression_probability",
            "model_valid", "calibration_status", "threshold",
            "threshold_ci_low", "threshold_ci_high", "n_controls",
            "control_contamination_fraction"
        )
        metric_columns = [
            f"{MARKER}__{suffix}"
            for suffix in metric_suffixes
            if f"{MARKER}__{suffix}" in evidence
        ]
        display(evidence.loc[:, ["object_id", *metric_columns]].head(20))
        display(evidence[f"{MARKER}__state"].value_counts(dropna=False).rename("cells").to_frame())
        if not calibration_audit.empty:
            pair_model = calibration_audit.loc[
                calibration_audit["donor_id"].astype(str).eq(DONOR)
                & calibration_audit["marker"].astype(str).eq(MARKER)
            ]
            display(pair_model)
    else:
        print(f"No calibrated marker columns are available for donor {DONOR}.")
else:
    print("No donor calibration has completed yet. Re-run from the progress cell later.")

## Selected donor-marker evidence diagnostics

These plots use analysis-eligible cells to show the downstream impact of one donor-marker calibration. The state bar emphasizes indeterminate and unavailable evidence. The threshold-relative histogram shows the bootstrap ambiguity band around zero; state colors also reveal cells that remain indeterminate because the empirical-tail criterion disagrees with threshold separation. Distribution shape and positive fraction are descriptive only and must not be used to tune calibration.

In [ ]:
STATE_ORDER = ("negative", "indeterminate", "positive", "unavailable")
STATE_COLORS = {
    "negative": "#457b9d",
    "indeterminate": "#f4a261",
    "positive": "#2a9d8f",
    "unavailable": "#bdbdbd",
}

if DONOR is None or MARKER is None:
    print("No completed donor-marker calibration is available to plot yet.")
else:
    state_column = f"{MARKER}__state"
    ratio_column = f"{MARKER}__log2_threshold_ratio"
    eligible_evidence = evidence.loc[evidence["qc_analysis_eligible"].astype(bool)].copy()
    state_counts = (
        eligible_evidence[state_column].astype("object").value_counts(dropna=False)
        .reindex(STATE_ORDER, fill_value=0)
    )
    state_fractions = state_counts / max(1, int(state_counts.sum()))
    pair_model = calibration_audit.loc[
        calibration_audit["donor_id"].astype(str).eq(DONOR)
        & calibration_audit["marker"].astype(str).eq(MARKER)
    ]
    if len(pair_model) != 1:
        raise ContractError(
            f"expected one calibration model for {DONOR}/{MARKER}, found {len(pair_model)}"
        )
    model = pair_model.iloc[0]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    bars = axes[0].bar(
        STATE_ORDER,
        state_fractions.to_numpy(dtype=float),
        color=[STATE_COLORS[name] for name in STATE_ORDER],
    )
    axes[0].set_ylim(0, max(1.0, 1.12 * float(state_fractions.max())))
    axes[0].set_ylabel("Fraction of analysis-eligible cells")
    axes[0].set_title("Evidence disposition (descriptive)")
    axes[0].tick_params(axis="x", rotation=20)
    axes[0].grid(axis="y", alpha=0.2)
    for bar, fraction, count in zip(bars, state_fractions, state_counts, strict=True):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{100 * fraction:.1f}%\n({int(count):,})",
            ha="center", va="bottom", fontsize=8,
        )

    ratios = pd.to_numeric(eligible_evidence[ratio_column], errors="coerce").to_numpy(dtype=float)
    finite_ratios = ratios[np.isfinite(ratios)]
    if finite_ratios.size:
        lower, upper = np.quantile(finite_ratios, [0.005, 0.995])
        if not np.isfinite(lower) or not np.isfinite(upper) or lower == upper:
            lower, upper = float(np.nanmin(finite_ratios)), float(np.nanmax(finite_ratios))
        if lower == upper:
            lower, upper = lower - 0.5, upper + 0.5
        bins = np.linspace(lower, upper, 81)
        centers = (bins[:-1] + bins[1:]) / 2
        widths = np.diff(bins)
        bottom = np.zeros(len(centers), dtype=float)
        finite_total = int(finite_ratios.size)
        states = eligible_evidence[state_column].astype("object").to_numpy()
        for state_name in ("negative", "indeterminate", "positive"):
            values = ratios[(states == state_name) & np.isfinite(ratios)]
            counts, _ = np.histogram(np.clip(values, lower, upper), bins=bins)
            fractions = counts / finite_total
            axes[1].bar(
                centers, fractions, width=widths, bottom=bottom,
                color=STATE_COLORS[state_name], label=state_name, linewidth=0,
            )
            bottom += fractions
        axes[1].axvline(0, color="black", linewidth=1.2, label="fitted threshold")
        if model["status"] == "valid":
            ci_low_ratio = (model["threshold_ci_low_log1p"] - model["threshold_log1p"]) / np.log(2.0)
            ci_high_ratio = (model["threshold_ci_high_log1p"] - model["threshold_log1p"]) / np.log(2.0)
            axes[1].axvspan(
                ci_low_ratio, ci_high_ratio, color="#ffd166", alpha=0.25,
                label="bootstrap ambiguity band",
            )
        axes[1].set_xlim(lower, upper)
        axes[1].set_xlabel("Log2 threshold ratio (tails clipped at 0.5th/99.5th percentiles)")
        axes[1].set_ylabel("Fraction of eligible measurable cells per bin")
        axes[1].set_title("Threshold-relative evidence by state")
        axes[1].grid(axis="y", alpha=0.2)
        axes[1].legend(fontsize=8)
    else:
        axes[1].text(
            0.5, 0.5,
            f"No finite threshold-relative scores\nmodel status: {model['status']}",
            ha="center", va="center", transform=axes[1].transAxes,
        )
        axes[1].set_axis_off()
    fig.suptitle(
        f"Donor {DONOR} · {MARKER} (reference: {model['reference']}; status: {model['status']})"
    )
    plt.show()

## Handoff

Proceed to typing only after reviewing invalid and indeterminate donor-marker models—not just positive-cell counts. Hierarchical typing consumes calibrated evidence with its uncertainty intact, so missing or unreliable markers remain explicit rather than silently becoming negatives.